# All Function

In [3]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.optimize import minimize
import os
np.random.seed(42)
# Step 1: Load Initial Data
def load_data(function_number, base_path):
    inputs_path = os.path.join(base_path, f'function_{function_number}', 'initial_inputs.npy')
    outputs_path = os.path.join(base_path, f'function_{function_number}', 'initial_outputs.npy')
    
    inputs = np.load(inputs_path)
    outputs = np.load(outputs_path)

    return inputs, outputs

# Step 2: Define the Gaussian Process Model
def define_gp_model():
    kernel = Matern(nu=2.5)
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, random_state=42)
    return gp

# Step 3: Fit the Model
def fit_model(gp, inputs, outputs):
    gp.fit(inputs, outputs)
    return gp

# Step 4: Define the Acquisition Function (Upper Confidence Bound)
def acquisition_function(x, model, kappa=2.0):
    x = np.atleast_2d(x)
    mean, std = model.predict(x, return_std=True)
    return -(mean + kappa * std)  # Negative for minimization

# Step 5: Optimize the Acquisition Function
def suggest_next_point(bounds, model):
    result = minimize(lambda x: acquisition_function(x, model),
                      x0=np.random.uniform(bounds[:, 0], bounds[:, 1]),
                      bounds=bounds, method='L-BFGS-B')
    return result.x

# Step 6: Format the Output
def format_output(point):
    return '-'.join([f"{x:.6f}" for x in point])

# Main Execution
def run_pipeline(function_number, base_path):
    inputs, outputs = load_data(function_number, base_path)
    gp = define_gp_model()
    gp = fit_model(gp, inputs, outputs)
    bounds = np.array([[0.000001, 0.999999]] * inputs.shape[1])  # Assuming input ranges [0,1]
    next_point = suggest_next_point(bounds, gp)
    formatted_output = format_output(next_point)
    print(f"Function {function_number} Suggested next point: {formatted_output}")

# Example usage
base_path = r"C:\Users\KarthikVenkatachalam\Downloads\Capstone Project\IMP-PCMLAI-capstone-initial_data\initial_data"

for i in range(1, 9):
    run_pipeline(i, base_path)


Function 1 Suggested next point: 0.000001-0.999999
Function 2 Suggested next point: 0.999999-0.000001
Function 3 Suggested next point: 0.531095-0.000001-0.000001
Function 4 Suggested next point: 0.000001-0.000001-0.000001-0.000001
Function 5 Suggested next point: 0.334576-0.346697-0.584829-0.135742
Function 6 Suggested next point: 0.000001-0.000001-0.282464-0.999999-0.000001
Function 7 Suggested next point: 0.000001-0.010468-0.226138-0.000001-0.221965-0.999999
Function 8 Suggested next point: 0.082510-0.286651-0.012878-0.236674-0.531776-0.490011-0.169890-0.474930
